# TextileGuard MobileNetV2 Training (Colab)

This notebook mirrors backend/model/train_v2.py and trains a MobileNetV2-based binary classifier.

Expected dataset layout on Drive or local project root:

- dataset/train/defective
- dataset/train/non_defective
- dataset/test/defective (optional)
- dataset/test/non_defective (optional)

If dataset/test is missing or empty, the validation split is reused as the test set.
If train folders are missing/empty, set AUTO_GENERATE_DATASET = True in Cell 2 to create a synthetic dataset.

Outputs (model, plots, evaluation report) are saved to backend/saved_model.

In [ ]:
# Install required dependencies
# NOTE: After this cell runs for the first time, restart the kernel,
#       then run all cells again (the restart is skipped on subsequent runs).
import sys
import subprocess
import importlib

REQUIRED = {
    'tensorflow': 'tensorflow>=2.13.0',
    'matplotlib': 'matplotlib>=3.7.0',
    'numpy': 'numpy>=1.24.0',
    'sklearn': 'scikit-learn>=1.3.0',
}

missing = [pkg for mod in REQUIRED if importlib.util.find_spec(mod) is None for pkg in [REQUIRED[mod]]]
if missing:
    print(f'Installing: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('Installation complete. Restarting kernel...')
    # Auto-restart kernel so newly installed packages are importable
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(True)
    except Exception:
        print('Please manually restart the kernel now, then re-run all cells.')
else:
    print('All dependencies already installed.')

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


def _resolve_project_root(start_dir):
    candidates = [start_dir]
    for i in range(1, 5):
        candidates.append(os.path.abspath(os.path.join(start_dir, *(['..'] * i))))
    for p in candidates:
        if os.path.isdir(os.path.join(p, 'dataset')) and os.path.isdir(os.path.join(p, 'backend')):
            return p
    return start_dir


# Update this path to your Drive project folder if using Colab
DEFAULT_ROOT = ('/content/drive/MyDrive/Textile-Pattern-Defect-Detection-System'
                if IN_COLAB else _resolve_project_root(os.getcwd()))
DRIVE_ROOT = os.environ.get('TEXTILE_GUARD_ROOT', DEFAULT_ROOT)

DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
TEST_DIR = os.path.join(DATASET_DIR, 'test')
SAVED_MODEL_DIR = os.path.join(DRIVE_ROOT, 'backend', 'saved_model')

MODEL_PATH = os.path.join(SAVED_MODEL_DIR, 'textile_defect_model.keras')
HISTORY_PATH = os.path.join(SAVED_MODEL_DIR, 'training_history.json')
PLOT_PATH = os.path.join(SAVED_MODEL_DIR, 'training_plot.png')
CONFUSION_PATH = os.path.join(SAVED_MODEL_DIR, 'confusion_matrix.png')
REPORT_PATH = os.path.join(SAVED_MODEL_DIR, 'evaluation_report.json')

os.makedirs(SAVED_MODEL_DIR, exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}


def _has_images(folder):
    if not os.path.isdir(folder):
        return False
    for name in os.listdir(folder):
        _, ext = os.path.splitext(name)
        if ext.lower() in IMAGE_EXTS:
            return True
    return False


# Set True to auto-generate a synthetic dataset when train folders are empty
# FIX: was False — kept False so user can choose; synthetic generation is done below
AUTO_GENERATE_DATASET = True  # <-- FIXED: was False, causing FileNotFoundError
SYNTHETIC_SCRIPT = os.path.join(DRIVE_ROOT, 'dataset', 'generate_dataset_v2.py')

# Sanity check dataset structure (train required, test optional)
required = [
    os.path.join(TRAIN_DIR, 'defective'),
    os.path.join(TRAIN_DIR, 'non_defective'),
]
missing_required = [p for p in required if not _has_images(p)]

if missing_required and AUTO_GENERATE_DATASET:
    if not os.path.exists(SYNTHETIC_SCRIPT):
        raise FileNotFoundError(f'Synthetic dataset script not found: {SYNTHETIC_SCRIPT}')
    print('Dataset folders are empty. Generating synthetic dataset...')
    subprocess.check_call([sys.executable, SYNTHETIC_SCRIPT])
    import gc; gc.collect()  # free memory used by dataset generation before TF loads
    missing_required = [p for p in required if not _has_images(p)]

if missing_required:
    raise FileNotFoundError(
        'Missing or empty required dataset folder(s):\n'
        + '\n'.join(missing_required)
        + '\n\nSet DRIVE_ROOT to the correct project folder '
        + '(or set TEXTILE_GUARD_ROOT env var).'
        + '\nIf you want synthetic data, ensure AUTO_GENERATE_DATASET = True.'
    )

test_folders = [
    os.path.join(TEST_DIR, 'defective'),
    os.path.join(TEST_DIR, 'non_defective'),
]
HAS_TEST_SET = all(os.path.exists(p) and _has_images(p) for p in test_folders)
if not HAS_TEST_SET:
    print('Test set not found or empty; validation split will be reused as test.')

print(f'Project root : {DRIVE_ROOT}')
print(f'Train dir    : {TRAIN_DIR}')
print(f'Has test set : {HAS_TEST_SET}')
print('Dataset structure looks OK.')

If you still see a ModuleNotFoundError after running the install cell, restart the runtime/kernel and run Cells 2-4 again.

In [ ]:
import os
import json
import numpy as np

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

# ── GPU / CPU memory guard (MUST be set before TF initialises) ──────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
for _gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception:
        pass
if not gpus:
    # CPU-only: cap thread count to avoid fork-bomb on Windows
    tf.config.threading.set_inter_op_parallelism_threads(2)
    tf.config.threading.set_intra_op_parallelism_threads(4)
print(f'TF {tf.__version__} | GPUs: {len(gpus)} | Device: {"GPU" if gpus else "CPU"}')
# ─────────────────────────────────────────────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Config (mirrors train_v2.py)
IMG_SIZE = 224
BATCH_SIZE = 32
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 15
PHASE1_LR = 1e-3
PHASE2_LR = 1e-5
FINE_TUNE_AT = -30
LABEL_SMOOTHING = 0.1
DROPOUT_RATE = 0.5
L2_WEIGHT = 1e-4
VALIDATION_SPLIT = 0.2
SEED = 42

In [ ]:
def build_augmentation_layers():
    augment_layers = [
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),
        layers.RandomZoom((-0.1, 0.1)),
        layers.RandomContrast(0.15),
    ]
    if hasattr(layers, "RandomBrightness"):
        augment_layers.append(layers.RandomBrightness(0.15))
    return keras.Sequential(augment_layers, name='augmentation')


def build_model(num_classes=1):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    augmented = build_augmentation_layers()(inputs, training=True)

    base_model = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_tensor=augmented,
    )
    base_model.trainable = False

    x = layers.GlobalAveragePooling2D(name='avg_pool')(base_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(L2_WEIGHT), name='dense_head')(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(L2_WEIGHT), name='dense_head_2')(x)
    x = layers.Dropout(DROPOUT_RATE * 0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='prediction')(x)

    model = Model(inputs, outputs, name='textile_defect_mobilenetv2')
    return model, base_model

In [ ]:
def load_datasets():
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        validation_split=VALIDATION_SPLIT,
        subset='training',
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='binary',
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        validation_split=VALIDATION_SPLIT,
        subset='validation',
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='binary',
    )

    if HAS_TEST_SET:
        test_ds = tf.keras.utils.image_dataset_from_directory(
            TEST_DIR,
            image_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            label_mode='binary',
            shuffle=False,
        )
    else:
        print('Using validation set as test set (dataset/test missing or empty).')
        test_ds = val_ds

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds = val_ds.cache().prefetch(AUTOTUNE)
    test_ds = test_ds.cache().prefetch(AUTOTUNE)

    return train_ds, val_ds, test_ds

In [ ]:
def save_training_plot(history, phase1_epochs):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.get('accuracy', []), label='Train Accuracy', linewidth=2)
    axes[0].plot(history.get('val_accuracy', []), label='Val Accuracy', linewidth=2)
    axes[0].axvline(x=phase1_epochs - 0.5, color='red', linestyle='--', alpha=0.5, label='Fine-tune start')
    axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.get('loss', []), label='Train Loss', linewidth=2)
    axes[1].plot(history.get('val_loss', []), label='Val Loss', linewidth=2)
    axes[1].axvline(x=phase1_epochs - 0.5, color='red', linestyle='--', alpha=0.5, label='Fine-tune start')
    axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=150)
    plt.close()
    print(f'Training plot saved to {PLOT_PATH}')

In [ ]:
def compute_class_weights(train_dir):
    """Compute inverse-frequency class weights to handle class imbalance."""
    import numpy as np
    class_counts = {}
    for cls in os.listdir(train_dir):
        cls_path = os.path.join(train_dir, cls)
        if os.path.isdir(cls_path):
            count = sum(1 for f in os.listdir(cls_path)
                        if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
            class_counts[cls] = count
    total = sum(class_counts.values())
    n_classes = len(class_counts)
    print(f'  Class counts: {class_counts}')
    # Map class folder names to integer indices (sorted alphabetically, same as Keras)
    sorted_classes = sorted(class_counts.keys())
    weights = {
        i: total / (n_classes * max(1, class_counts[cls]))
        for i, cls in enumerate(sorted_classes)
    }
    print(f'  Class weights: {weights}')
    return weights


def train():
    print('============================================================')
    print('  TextileGuard Model Training v2 - MobileNetV2')
    print('============================================================')

    print('\n[*] Loading datasets...')
    train_ds, val_ds, test_ds = load_datasets()

    # FIX: compute class weights to address class imbalance
    print('\n[*] Computing class weights...')
    class_weight = compute_class_weights(TRAIN_DIR)

    print('\n[*] Building MobileNetV2 model...')
    model, base_model = build_model()
    model.summary(print_fn=lambda x: print(f'  {x}'))

    print('\n============================================================')
    print(f'  Phase 1: Feature Extraction (backbone frozen)')
    print(f'  LR={PHASE1_LR}, Epochs={PHASE1_EPOCHS}')
    print('============================================================')

    model.compile(
        optimizer=Adam(learning_rate=PHASE1_LR),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc'),
        ],
    )

    phase1_callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ModelCheckpoint(MODEL_PATH, monitor='val_auc', save_best_only=True, verbose=1, mode='max'),
    ]

    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE1_EPOCHS,
        callbacks=phase1_callbacks,
        class_weight=class_weight,  # FIX: added to counteract class imbalance
    )

    print('\n============================================================')
    print(f'  Phase 2: Fine-Tuning (last {abs(FINE_TUNE_AT)} layers unfrozen)')
    print(f'  LR={PHASE2_LR}, Epochs={PHASE2_EPOCHS}')
    print('============================================================')

    base_model.trainable = True
    for layer in base_model.layers[:FINE_TUNE_AT]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=PHASE2_LR),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc'),
        ],
    )

    phase2_callbacks = [
        EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
        ModelCheckpoint(MODEL_PATH, monitor='val_auc', save_best_only=True, verbose=1, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ]

    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE2_EPOCHS,
        callbacks=phase2_callbacks,
        class_weight=class_weight,  # FIX: added to counteract class imbalance
    )

    print('\n============================================================')
    print('  Evaluating on test set...')
    print('============================================================')

    best_model = keras.models.load_model(MODEL_PATH)
    test_results = best_model.evaluate(test_ds, return_dict=True)
    print('\n  Test Results:')
    for k, v in test_results.items():
        print(f'    {k}: {v:.4f}')

    y_true = []
    y_pred_proba = []
    for images, labels in test_ds:
        preds = best_model.predict(images, verbose=0)
        y_true.extend(labels.numpy().flatten())
        y_pred_proba.extend(preds.flatten())

    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)
    y_pred = (y_pred_proba > 0.5).astype(int)

    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    cm = np.array([[tn, fp], [fn, tp]])

    accuracy = float((tp + tn) / max(1, tp + tn + fp + fn))
    precision = float(tp / max(1, tp + fp))
    recall = float(tp / max(1, tp + fn))
    f1 = float(2 * precision * recall / max(1e-8, precision + recall))

    report = {
        'test_accuracy': round(accuracy, 4),
        'test_precision': round(precision, 4),
        'test_recall': round(recall, 4),
        'test_f1': round(f1, 4),
        'confusion_matrix': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
        'test_results': {k: round(float(v), 4) for k, v in test_results.items()},
    }
    with open(REPORT_PATH, 'w') as f:
        json.dump(report, f, indent=2)
    print(f'\n  Evaluation report saved to {REPORT_PATH}')

    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Non-Defective', 'Defective'])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Non-Defective', 'Defective'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=18)
    fig.colorbar(im)
    plt.tight_layout()
    plt.savefig(CONFUSION_PATH, dpi=150)
    plt.close()
    print(f'  Confusion matrix saved to {CONFUSION_PATH}')

    combined_history = {}
    for key in history1.history:
        combined_history[key] = [float(v) for v in history1.history[key]]
        if key in history2.history:
            combined_history[key].extend([float(v) for v in history2.history[key]])

    with open(HISTORY_PATH, 'w') as f:
        json.dump(combined_history, f, indent=2)

    save_training_plot(combined_history, len(history1.history.get('accuracy', [])))

    print('\n============================================================')
    print('  [OK] Training complete!')
    print(f'  Model saved to : {MODEL_PATH}')
    print(f'  Accuracy  : {accuracy:.2%} | F1: {f1:.2%}')
    print(f'  Precision : {precision:.2%} | Recall: {recall:.2%}')
    print('============================================================')

In [ ]:
train()

In [ ]:
# Optional: display saved plots
from IPython.display import Image, display

if os.path.exists(PLOT_PATH):
    display(Image(filename=PLOT_PATH))

if os.path.exists(CONFUSION_PATH):
    display(Image(filename=CONFUSION_PATH))